# Massey ranking

Description: Construct a Massey ranking of data.

### Set parameters

gameFilename - game data file, presumed to be in the format from 
the Massey rating data server, which can be found at  

teamFilename - team data file

k - number of teams to print in the final ranking - set to 0 to get all teams

In [2]:
gameFilename = '24games.txt'
teamFilename = '24teams.txt'
k = 20

### Load the team names into an array

In [6]:
import pandas as pd

teamNames = pd.read_csv(teamFilename, header = None)
numTeams = len(teamNames)

### Load the games

In [9]:
# columns of games are:
#	column 0 = days since 1/1/0000
#	column 1 = date in YYYYMMDD format
#	column 2 = team1 index
#	column 3 = team1 homefield (1 = home, -1 = away, 0 = neutral)
#	column 4 = team1 score
#	column 5 = team2 index
#	column 6 = team2 homefield (1 = home, -1 = away, 0 = neutral)
#	column 7 = team2 score
games = pd.read_csv(gameFilename, header = None)
numGames = len(games)

### Create the Massey linear system

In [12]:
import numpy as np

masseyMatrix = np.zeros((numTeams,numTeams))
b = np.zeros(numTeams)

for i in range(numGames):
    team1ID = games.loc[i, 2] - 1 # subtracting 1 since python indexes at 0
    team1Score = games.loc[i, 4]
    team2ID = games.loc[i, 5] - 1 # subtracting 1 since python indexes at 0
    team2Score = games.loc[i, 7]
    
    masseyMatrix[team1ID, team2ID] -= 1
    masseyMatrix[team2ID, team1ID] -= 1

    masseyMatrix[team1ID, team1ID] += 1
    masseyMatrix[team2ID, team2ID] += 1
    
    pointDifferential = abs(team1Score - team2Score)
    
    if team1Score > team2Score:
        b[team1ID] += pointDifferential
        b[team2ID] -= pointDifferential
    elif team1Score < team2Score:
        b[team1ID] -= pointDifferential
        b[team2ID] += pointDifferential
        
# replace last row with ones and 0 on RHS
masseyMatrix[-1,:] = np.ones((1,numTeams))
b[-1] = 0

### Calculate linear system

In [15]:
r = np.linalg.solve(masseyMatrix,b)
iSort = np.argsort(-r)
iSort

array([  9,  14,   8,   2,  10,  13,  15,   5,   4,  16,   1,  17,  18,
        19,  20,  21,  22,  23,   7,  24,  25,  26,  27,  31,  28,  30,
        29,  34,  35,  12,  33,  36,  32,  37,  39,  38,  11,  40,  41,
        43,  42,  44,  45,  46,  48,  49,  47,  52,  51,  50,  53,  54,
        55,  56,  59,  57,  58,  60,  62,  63,  65,  66,  61,  67,  64,
        68,  70,  71,   6,  72,  76,  74,  73,  69,  75,  77,  78,  79,
        80,  82,  81,  83,  85,  84,  86,  87,  88,   3,  89,  90,  91,
        93,  92,  94,  95,  96,  98,  97, 100,  99, 101,   0])

### Print the ranking of the teams

In [18]:
print('\n\n************** MASSEY Rating Method **************\n')
print('===========================')
print('Rank   Rating    Team   ')
print('===========================')
if k==0:
    numberTeamToPrint = numTeams
else:
    numberTeamToPrint = k

for i in range(numberTeamToPrint):
    print(f'{i+1:4d}   {r[iSort[i]]:.5f}  {teamNames.loc[iSort[i],1]}')

print('')   # extra carriage return



************** MASSEY Rating Method **************

Rank   Rating    Team   
   1   2.75118   10
   2   2.75118   15
   3   2.70521   9
   4   2.59026   3
   5   2.59026   11
   6   2.40636   14
   7   2.10980   Сахипов Амир
   8   2.07302   6
   9   2.01555   5
  10   1.97647   Шарел Марғұлан
  11   1.96957   2
  12   1.84314   Юркевич Мирон
  13   1.77647   Даутов Жан
  14   1.57647   Аманжолов Амирлан
  15   1.50980   Мурат Тимур
  16   1.30980   Маратұлы Әлинұр
  17   1.30980   Роговский Владимир
  18   1.24314   Таңсықбай Абзал
  19   1.08452   8
  20   1.04314   Нәрік Арсен



In [31]:
index = np.where(iSort == 0)[0][0]
print(index)
ProblemFile = open("Problem ranking Massey", "w")
for i in range(15):
    index = np.where(iSort == i)[0][0]
    ProblemFile.write('%3i,%3i, %f\n' %(i+1,index+1, r[iSort[index]]))
ProblemFile.close()

101


In [22]:
index = np.where(iSort == 9)[0][0]
print(index)
r[iSort[index]]

0


2.7511832319134557

### Calculate predictability of method

In [25]:
numberCorrectPredictions = 0
for i in range(numGames):
    team1ID = games.loc[i, 2] - 1 
    team1Score = games.loc[i, 4]
    team2ID = games.loc[i, 5] - 1 
    team2Score = games.loc[i, 7]
    
    if team1Score > team2Score and r[team1ID] > r[team2ID]:
        numberCorrectPredictions += 1
    elif team2Score > team1Score and r[team2ID] > r[team1ID]:
        numberCorrectPredictions += 1
    elif team1Score == team2Score and r[team1ID] == r[team2ID]:
        numberCorrectPredictions += 1

print(f'Predictability: {numberCorrectPredictions/numGames*100:.2f}%') 


Predictability: 89.35%


In [27]:
points=pd.read_csv('24points.txt', header = None)
for i in range(15):
    Psum=points.iloc[:,i+1].sum()
    print(Psum)

502
83
18
414
76
67
366
170
7
2
18
238
215
36
2
